# ML04 · PCA 降維

用主成分分析 principal component analysis（PCA）把高維特徵壓縮成少數幾個方向，在保留最多資訊的前提下，讓資料能畫成 2D 圖看出結構。

## 學習目標
- 理解降維 dimensionality reduction 的動機：特徵太多會帶來哪些困擾。
- 建立主成分 principal component 的直覺：資料變異最大的方向。
- 知道為什麼要先標準化 standardization 再做 PCA。
- 會用 sklearn 的 PCA 進行 fit 與 transform 求得降維結果。
- 會讀解釋變異比例 explained_variance_ratio_ 來判斷該保留幾維。
- 能把多維特徵壓到 2D 並視覺化 visualization，觀察群落與分布。

In [ ]:
# 概念：統一匯入套件並設定畫圖字型，讓中文標題不會變成方框
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 注意：matplotlib 預設字型不含中文，需指定一個系統有的中文字型
matplotlib.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "Microsoft YaHei", "SimHei", "Arial Unicode MS"]
matplotlib.rcParams["axes.unicode_minus"] = False   # 避免負號顯示成方框

print("套件載入完成")

## 為什麼要降維

降維 dimensionality reduction 是把「很多欄特徵」用「比較少的欄」重新表示，盡量不丟資訊。

為什麼要學：特徵太多時有三個困擾。
- 維度災難 curse of dimensionality：維度越高，資料越稀疏，模型越容易過擬合 overfitting。
- 特徵冗餘 feature redundancy：許多欄其實彼此高度相關 correlated features，重複描述同一件事。
- 無法視覺化：超過 3 維就畫不出來，看不見資料結構。

何時用：欄位多、彼此相關、想壓縮或想畫成 2D 圖探索時。

In [ ]:
# 概念：用 numpy 造彼此高度相關的欄位，直觀感受「資訊其實沒那麼多維」
rng = np.random.default_rng(0)   # 固定亂數種子，讓每次結果一致方便對照

n = 200
# 造一個共同的「規模」因子：每棟建築有多大，由它驅動其他欄位
scale = rng.normal(loc=0.0, scale=1.0, size=n)

# 樓高、樓層數、總面積其實都跟「規模」走，只是加了一點獨立雜訊
floors = 10 + 3 * scale + rng.normal(0, 0.3, n)          # 樓層數
height = 3.2 * floors + rng.normal(0, 1.0, n)            # 樓高約等於樓層數乘層高
area = 220 + 60 * scale + rng.normal(0, 5.0, n)          # 總面積也跟著規模走

data = np.column_stack([floors, height, area])           # 疊成 (200, 3) 的特徵矩陣
print("資料 shape:", data.shape)

# 相關係數矩陣：值接近 1 表示兩欄幾乎是同一件事的不同說法
corr = np.corrcoef(data, rowvar=False)                   # rowvar=False 表示每欄是一個變數
print("三欄之間的相關係數矩陣:")
print(np.round(corr, 3))

## 主成分的直覺

主成分 principal component 就是「資料散得最開的方向」。

- 第一主成分（first principal component）：變異量 variance 最大的方向，資料沿著它拉得最長。
- 第二主成分（second principal component）：在與第一主成分垂直（正交方向 orthogonal directions）的前提下，抓剩下變異最大的方向。

把點投影 projection 到主成分上，就得到降維後的新座標。重點是直覺：找一條軸，讓投影後的點盡量分得開、資訊損失最少。

In [ ]:
# 概念：在 2D 散點上畫出第一、第二主成分方向，看 PCA 抓到的就是最散的方向
rng = np.random.default_rng(1)

# 造一團沿著斜對角拉長的資料：樓高與面積正相關，所以呈斜橢圓分布
n = 150
base = rng.normal(0, 1, n)
height = 25 + 8 * base + rng.normal(0, 1.5, n)           # 樓高（公尺）
area = 150 + 70 * base + rng.normal(0, 12, n)            # 總面積（平方公尺）
points = np.column_stack([height, area])

# 先標準化，主成分方向才不會被面積的大數值主導（下一節詳述原因）
points_std = StandardScaler().fit_transform(points)

pca = PCA(n_components=2)
pca.fit(points_std)
center = points_std.mean(axis=0)                         # 箭頭從資料中心畫出
# components_ 每一列是一個主成分方向；用解釋變異開根號當箭頭長度，方向越主要越長
lengths = np.sqrt(pca.explained_variance_)

plt.figure(figsize=(5, 5))
plt.scatter(points_std[:, 0], points_std[:, 1], s=12, alpha=0.5, label="標準化後的點")
for i, (vec, length) in enumerate(zip(pca.components_, lengths)):
    plt.arrow(center[0], center[1], vec[0] * length * 2, vec[1] * length * 2,
              width=0.03, color="red" if i == 0 else "green",
              length_includes_head=True,
              label=f"第{i + 1}主成分")
plt.axis("equal")          # 等比例座標，才看得出兩軸正交
plt.legend()
plt.title("第一、第二主成分方向")
plt.xlabel("樓高（標準化）"); plt.ylabel("面積（標準化）")
plt.show()
print("兩主成分是否正交（內積接近 0）:", np.round(pca.components_[0] @ pca.components_[1], 6))

## 先標準化再 PCA

PCA 看的是變異量，而變異量會受單位影響。面積動輒上萬、容積率只有個位數，若不處理，PCA 幾乎只會看面積這一欄。

標準化 standardization 用 z-score 把每欄轉成「平均 0、標準差 1」，消除尺度差異 scale difference，讓每個特徵站在同一條起跑線上。sklearn 用 StandardScaler 做這件事。

公式：z = (x − 平均) / 標準差。何時用：各特徵單位不同時，做 PCA 前幾乎都要先標準化。

In [ ]:
# 概念：比較「未標準化」與「標準化後」第一主成分方向的差異
rng = np.random.default_rng(2)

n = 120
floors = rng.integers(3, 30, n).astype(float)            # 樓層數（個位到數十）
area = floors * 90 + rng.normal(0, 200, n) + 3000        # 面積（數千到上萬，數量級遠大於樓層）
raw = np.column_stack([floors, area])

# 未標準化：直接 PCA
pca_raw = PCA(n_components=2).fit(raw)
# 標準化後：先縮放再 PCA
scaled = StandardScaler().fit_transform(raw)
pca_scaled = PCA(n_components=2).fit(scaled)

# 第一主成分在 [樓層, 面積] 上的權重；數值大表示該欄主導這個方向
print("未標準化 第一主成分權重 [樓層, 面積]:", np.round(pca_raw.components_[0], 3))
print("標準化後 第一主成分權重 [樓層, 面積]:", np.round(pca_scaled.components_[0], 3))
# 注意：未標準化時面積權重幾乎是 1，等於只看面積；標準化後兩欄貢獻才均衡
print("未標準化 解釋變異比例:", np.round(pca_raw.explained_variance_ratio_, 3))
print("標準化後 解釋變異比例:", np.round(pca_scaled.explained_variance_ratio_, 3))

## 用 sklearn 跑 PCA：fit 與 transform

sklearn 的 PCA 是一個估計器 estimator，用法與其他模型一致。
- fit：從資料學出主成分方向（找出那幾條軸）。
- transform：把資料投影到這些軸上，得到降維後座標 transformed coordinates。
- n_components：要保留幾個主成分（降到幾維）。

形狀：
```
pca = PCA(n_components=k)
pca.fit(X)            # 學方向，X 為 (N, D)
Z = pca.transform(X)  # 投影，Z 為 (N, k)
```
fit 與 transform 可合併成 fit_transform。

In [ ]:
# 概念：對標準化後的多欄建築資料做 PCA(n_components=2) 的 fit 與 transform
rng = np.random.default_rng(3)

n = 100
# 造五欄建築資料：樓高、樓層、面積、屋齡、距捷運距離
factor = rng.normal(0, 1, n)                             # 共同規模因子
height = 30 + 9 * factor + rng.normal(0, 2, n)
floors = 9 + 2.8 * factor + rng.normal(0, 0.5, n)
area = 250 + 80 * factor + rng.normal(0, 10, n)
age = rng.uniform(0, 40, n)                              # 屋齡與規模無關，較獨立
dist_mrt = rng.uniform(50, 1500, n)                      # 距捷運距離，也較獨立
X = np.column_stack([height, floors, area, age, dist_mrt])
print("原始特徵 shape:", X.shape)

X_std = StandardScaler().fit_transform(X)                # PCA 前先標準化

pca = PCA(n_components=2)
pca.fit(X_std)                                           # 從資料學出兩個主成分方向
Z = pca.transform(X_std)                                 # 投影到這兩個方向
print("降維後座標 shape:", Z.shape)                        # 應為 (100, 2)
print("前三筆降維後座標:")
print(np.round(Z[:3], 3))

## 解釋變異比例：該留幾維

解釋變異比例 explained_variance_ratio_ 告訴你每個主成分保留了原始資料多少比例的變異。

把它從大到小累加，得到累積解釋變異 cumulative explained variance。常見做法是設一個門檻（例如 90%），累積到超過門檻的最少維數，就是要保留的維度數，兼顧壓縮與資訊保留量。

何時用：決定 n_components 要設多少時，用這條曲線來判斷，而不是憑感覺。

In [ ]:
# 概念：列出各主成分解釋變異比例並畫累積曲線，判斷壓到 2 維保留多少資訊
rng = np.random.default_rng(4)

n = 200
factor = rng.normal(0, 1, n)
# 六欄：前幾欄受同一規模因子驅動（高度相關），後幾欄較獨立
f1 = factor + rng.normal(0, 0.2, n)
f2 = 2 * factor + rng.normal(0, 0.2, n)
f3 = -1.5 * factor + rng.normal(0, 0.2, n)
f4 = rng.normal(0, 1, n)
f5 = rng.normal(0, 1, n)
f6 = 0.5 * f4 + rng.normal(0, 0.3, n)
X = np.column_stack([f1, f2, f3, f4, f5, f6])

X_std = StandardScaler().fit_transform(X)
pca = PCA().fit(X_std)                                   # 不指定 n_components 會保留全部，方便看每維貢獻

ratio = pca.explained_variance_ratio_
cum = np.cumsum(ratio)                                   # 從大到小累加得到累積解釋變異
print("各主成分解釋變異比例:", np.round(ratio, 3))
print("累積解釋變異:", np.round(cum, 3))
print("壓到 2 維保留的資訊:", f"{cum[1]:.1%}")

plt.figure(figsize=(6, 4))
xs = np.arange(1, len(ratio) + 1)
plt.bar(xs, ratio, alpha=0.5, label="各主成分解釋變異")
plt.plot(xs, cum, "o-", color="red", label="累積解釋變異")
plt.axhline(0.9, color="gray", linestyle="--", label="90% 門檻")   # 門檻線輔助判斷
plt.xlabel("主成分序號"); plt.ylabel("解釋變異比例")
plt.title("解釋變異比例與累積曲線")
plt.legend()
plt.show()

## 壓到 2D 並視覺化

把多維資料投影到前兩個主成分後，就能用一張 2D 散點圖 scatter plot 看出結構：群落、離群點、類別分離程度。

這張圖的兩軸是主成分而非原始特徵，可看成資料的潛在空間 latent space：方向本身沒有單位意義，但相對位置反映樣本間的相似度。依類別上色後，若同類聚成一團、不同類分得開，代表這些特徵確實能區分類別。

何時用：探索資料、檢查類別是否可分、為後續分群做準備時。

In [ ]:
# 概念：把多維建築資料壓到 2D，依用途類別（住宅／商辦）上色觀察是否自然分開
rng = np.random.default_rng(5)

def make_buildings(center, n, label):
    # 造同一類別的建築：各欄圍繞各自的中心抖動，模擬同類風格相近
    height = rng.normal(center[0], 3, n)
    floors = rng.normal(center[1], 1, n)
    area = rng.normal(center[2], 15, n)
    far = rng.normal(center[3], 0.3, n)                  # 容積率 floor area ratio
    X = np.column_stack([height, floors, area, far])
    y = np.full(n, label)
    return X, y

# 住宅：較矮、面積中等；商辦：較高、面積大、容積率高
X_res, y_res = make_buildings([28, 9, 230, 2.2], 80, 0)   # 0 = 住宅
X_off, y_off = make_buildings([55, 18, 420, 4.5], 80, 1)  # 1 = 商辦
X = np.vstack([X_res, X_off])
y = np.concatenate([y_res, y_off])

X_std = StandardScaler().fit_transform(X)
Z = PCA(n_components=2).fit_transform(X_std)             # 一步到位：學方向並投影到 2D

plt.figure(figsize=(6, 5))
for label, name, color in [(0, "住宅", "tab:blue"), (1, "商辦", "tab:orange")]:
    mask = y == label                                    # 布林遮罩挑出該類別的點
    plt.scatter(Z[mask, 0], Z[mask, 1], s=18, alpha=0.6, color=color, label=name)
plt.xlabel("第一主成分"); plt.ylabel("第二主成分")
plt.title("建築資料壓到 2D 後依用途上色")
plt.legend()
plt.show()
print("降維後 shape:", Z.shape, " 兩類別是否在圖上分開，可用肉眼觀察")

## 練習

以下三題由淺入深，皆為複合型但簡短。資料請自己用 numpy 造（仿真即可），完成後對照「驗收」確認輸出。

In [ ]:
# TODO 1 ·（簡單）把相關特徵壓成一條軸（整合：相關特徵 + 主成分 + fit/transform）
#   情境：你有一批建築資料，樓高與樓層數幾乎成正比，想用一個數字代表「整體規模」。
#   要求：
#     1. 用 numpy 自造高度相關的樓高與樓層數兩欄資料（讓樓高約等於樓層數乘層高再加雜訊）。
#     2. 對其做 PCA(n_components=1) 的 fit/transform，取得單一主成分座標。
#     3. 印出第一主成分的解釋變異比例。
#   驗收：應看到兩欄被壓成一條「規模軸」(N, 1)，且第一主成分解釋了絕大部分變異（接近 1）。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）標準化前後的差異（整合：標準化 + PCA + 解釋變異比例 + 視覺化）
#   情境：建築資料含面積（數值很大）與容積率（數值很小），想看尺度是否影響降維。
#   要求：
#     1. 自造含 面積、容積率、樓層數 的多欄資料（面積數千、容積率個位數）。
#     2. 分別在「未標準化」與「StandardScaler 標準化後」各跑一次 PCA 壓到 2D。
#     3. 比較兩者的 explained_variance_ratio_，並各畫一張 2D 散點圖。
#   驗收：應看到未標準化時面積主導第一主成分（解釋比例極高），
#         標準化後各特徵貢獻較均衡、群落分布更合理。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）該保留幾維與群落觀察（整合：累積解釋變異 + 維度選擇 + 2D 視覺化 + 潛在空間直覺）
#   情境：你有一份較多欄位的都市建築資料（樓高、面積、容積率、屋齡、距捷運距離等），
#         含兩種用途類別，想為後續分群做準備。
#   要求：
#     1. 自造約五到六欄、帶兩個隱含群落的建築資料（兩類別各欄中心略有差異）。
#     2. 標準化後跑 PCA，畫累積解釋變異曲線，並說明要保留幾維才達 90%。
#     3. 取前兩個主成分畫 2D 散點並依類別上色，討論潛在空間中兩類是否分離。
#   驗收：應看到累積曲線指出的維度數，以及 2D 圖中兩類別大致分群，為下一步的分群鋪路。
# TODO: 在這裡完成


## 小結

- 降維 dimensionality reduction 的動機：特徵太多會帶來維度災難、特徵冗餘與無法視覺化；許多欄其實高度相關，可用更少方向描述。
- 主成分 principal component 是資料散得最開的方向：第一主成分抓最大變異，後續主成分在正交方向抓剩餘變異，投影到主成分即得降維座標。
- PCA 看變異量，會被大尺度特徵主導，所以做 PCA 前要先用標準化（z-score / StandardScaler）讓各欄尺度可比較。
- sklearn 的 PCA 用 fit 學主成分方向、transform 投影得到新座標，n_components 決定降到幾維，流程與其他估計器一致。
- 解釋變異比例 explained_variance_ratio_ 顯示每維保留多少資訊；把它累加到門檻（如 90%）可決定該保留幾維。
- 把資料壓到前兩個主成分即可用一張 2D 散點圖觀察群落與類別分離，是探索資料與為分群做準備的利器。